# 第12课 - 使用代理备忘录减少聊天历史

本笔记本演示了如何使用微软代理框架管理长对话中的上下文。随着对话增长，令牌数量增加——最终超出模型的上下文窗口。我们通过<strong>上下文摘要模式</strong>和用于持久记忆的<strong>代理备忘录</strong>来解决这个问题。

## 你将学到：
1. <strong>为什么上下文管理重要</strong>：理解令牌限制和上下文窗口
2. <strong>上下文感知代理</strong>：构建能管理自身对话上下文的代理
3. <strong>上下文摘要模式</strong>：使用工具精简对话历史
4. <strong>代理备忘录</strong>：在上下文缩减中依然持久的记忆

## 前置条件：
- 配置好环境变量的 Azure OpenAI 设置
- 对之前课程中基本代理概念的理解


## 安装


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## 为什么上下文管理很重要

每个大型语言模型 (LLM) 都有一个有限的<strong>上下文窗口</strong>——即它在一次请求中能够处理的最大令牌数量。随着多轮对话的进行：

- <strong>令牌数量随着每条用户消息和助手回复线性增长</strong>。
- <strong>提示令牌主导成本</strong>，因为整个历史每轮都会被重新发送。
- 最终对话<strong>会超出上下文窗口</strong>，模型要么截断对话，要么报错。

### 管理上下文的策略

| 策略 | 工作原理 | 权衡 |
|---|---|---|
| <strong>截断</strong> | 删除最早的消息 | 失去早期的上下文 |
| <strong>总结</strong> | 将较旧的消息压缩成摘要 | 细节会丢失，但保留关键点 |
| **草稿板 / 外部记忆** | 将关键信息存储在对话外部 | 需要调用工具，但能够保存任何信息减少 |

在本笔记本中，我们将<strong>总结</strong>与<strong>草稿板工具</strong>结合，使代理即使在对话历史被压缩时也能保持连续性。


## 创建一个上下文感知代理


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## 模拟一次长对话

让我们通过一次多轮对话来看看上下文是如何积累的。代理应该在多轮对话中保留关键细节（偏好、预算、旅行日期）并展示连贯性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

注意代理如何保留早期对话的上下文——它知道关于日本、寿司、寺庙、摄影、3000美元预算、独自旅行和四月的时间范围。在短时间对话中这效果很好，但随着对话的增长，重新发送全部历史变得昂贵。

让我们继续对话更多回合，看看上下文的积累：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 上下文总结模式

随着对话的发展，我们可以使用<strong>总结工具</strong>将累积的上下文浓缩成简洁的格式。代理调用此工具记录关键偏好，以便即使较早的消息被删除，重要信息仍能被保留。

该模式是更复杂历史缩减的构建模块：
1. 代理识别对话中的关键事实
2. 它调用总结工具以持久化这些信息
3. 由于总结捕捉了重要内容，较早的消息可以安全删除

下面我们定义一个 `summarize_preferences` 工具，代理可以调用它来记录其所学内容的简洁摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 总结

在本课中，您学习了如何使用 Microsoft Agent Framework 管理长期运行代理对话中的上下文：

### 关键概念
- <strong>上下文窗口是有限的</strong> — 对话历史中的每个令牌都需要付费并计入限制。
- <strong>摘要工具</strong> 使代理能够将累积的上下文浓缩为简明的摘要，减少令牌使用量，同时保留关键信息。
- <strong>代理速记板</strong> 提供了持久的外部记忆，能保存任何对话缩减操作中遗失的内容。

### 您构建的内容
- 一个 <strong>上下文感知代理</strong> ，能在多轮对话中维持连续性
- 一个 <strong>摘要工具</strong> （`summarize_preferences`），以简洁格式记录关键用户细节
- 一个 <strong>多轮对话演示</strong>，展示上下文保留与变更处理

### 现实应用
- <strong>客服机器人</strong>：在长时间支持会话中记住偏好
- <strong>个人助理</strong>：跟踪进行中的项目，无需重复说明上下文
- <strong>教育导师</strong>：在多次互动中保持学生进度

### 后续步骤
- 实现一个带文件持久化的完整速记板工具
- 在摘要后添加自动历史截断功能
- 结合向量数据库实现语义记忆搜索
- 构建能够在数天后恢复对话并保持完整上下文的代理


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
